# EDA on the merged feature matrix

Run these first so `outputs/feature_matrix_master.csv` exists:
```
python scripts/run_ingest_gpovalues.py
python scripts/run_ingest_tier.py
python scripts/run_feature_build.py
```
This notebook is scaffolded for Phase 2-3 of ROADMAP.md. Values here are
real (gpovalues.com, solved from ~29K observed Discord trades) -- not a
personal trade log, per the project's current data collection approach.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config.settings import OUTPUT_DIR

df = pd.read_csv(OUTPUT_DIR / "feature_matrix_master.csv")
df.head()

## 1. Sanity check the tier-list enrichment join
How many gpovalues items got matched to a tier/category record? Which
ones didn't (worth checking if those are naming mismatches worth fixing
by hand, e.g. adding to the alias map)?

In [ ]:
df['has_tier_enrichment'].value_counts()

In [ ]:
df[~df['has_tier_enrichment']][['name', 'value', 'confidence']]

## 2. Confidence distribution
gpovalues flags confidence purely on trade count (<200 low, 200-999 medium,
1000+ high). Worth knowing up front how much of the catalog you're working
with is actually well-observed vs. thin.

In [ ]:
sns.countplot(data=df, x='confidence', order=['low', 'medium', 'high'])
plt.title('Item count by confidence band')

## 3. Value vs. tier/rarity
Does the community-solved value actually track the structural tier list, or
do they diverge? Divergence is interesting -- it's where the tier list is
stale or the market disagrees with the 'official' community ranking.

In [ ]:
enriched = df[df['has_tier_enrichment']]
sns.boxplot(data=enriched, x='tier_ordinal', y='value')
plt.yscale('log')
plt.title('Solved value by tier ordinal (log scale)')

## 4. Trend, once you have 2+ snapshot dates
Will print a message instead of results if you don't have enough history
yet -- that's expected in the first few days of running the ingestion script.

In [ ]:
from market_signals.models.trend_model import load_snapshot_history, compute_trend, MIN_SNAPSHOTS

history = load_snapshot_history()
n_dates = history['snapshot_date'].nunique()
print(f"{n_dates} snapshot date(s) so far (need {MIN_SNAPSHOTS}+ for a trend).")